# Phase E — Sentiment (CAMeLBERT) — Separate Clean Runtime

**Scope:** inference-only evaluation of `CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment` on the frozen 108-row `human_val` split. Uses the existing, unmodified `run_model_selection.py` / `hf_loading.py` / `sentiment_pretrained.py` code — no manual `torch.load` bypass, no code changes.

**This notebook is a brand-new, separate Colab runtime.** It never touches the runtime where MARBERTv2 Intent/Topic were trained (torch 2.5.1+cu121, verified). If you still have that notebook open, do not run any of these cells there.

**Guardrails baked into the commands below:**
- `--gold-csv data/gold/gold_train_val_only.csv` is always passed explicitly (this file has zero `human_test` rows — the full `gold_dataset_round1.csv` isn't even packaged in this bundle, so there is no accidental fallback path to it).
- `--skip-classical --skip-transformers` — only the pretrained sentiment leg runs. No MARBERTv2/AraBERT fine-tuning call happens (verified against the script: that block is fully gated behind `if not skip_transformers`).
- Taxonomy, splits, and `hf_loading.py`'s `use_safetensors=True` default are untouched. The only thing this notebook changes is which `torch` is installed.

**Precondition:** upload `rcs_colab_training_bundle.zip` to `/content/` before running Cell 1. Reuse the exact zip already built for the MARBERT run if you still have it (its `data/gold/gold_train_val_only.csv` sha256 is pinned below) rather than rebuilding, so both legs are provably running against byte-identical val data.

In [ ]:
# Cell 1 — Extract bundle + verify frozen val CSV integrity (no installs yet)
import hashlib
import json
import os
from pathlib import Path

ZIP = Path("/content/rcs_colab_training_bundle.zip")
if not ZIP.exists():
    raise FileNotFoundError("Upload rcs_colab_training_bundle.zip to /content/ first")

!unzip -q -o {ZIP} -d /content/
WORK = Path("/content/rcs_project")
os.chdir(WORK)

manifest = json.loads(Path("data/gold/bundle_training_manifest.json").read_text())
gold_csv = Path("data/gold/gold_train_val_only.csv")
actual_sha256 = hashlib.sha256(gold_csv.read_bytes()).hexdigest()
expected_sha256 = manifest["train_val_csv_sha256"]

print("expected val rows:", manifest["supervised_val_rows"], "| human_test excluded:", manifest["human_test_rows_excluded"])
print("sha256 match:", actual_sha256 == expected_sha256)
assert actual_sha256 == expected_sha256, "gold_train_val_only.csv does not match the manifest — do not proceed"
assert manifest["supervised_val_rows"] == 108, manifest["supervised_val_rows"]

In [ ]:
# Cell 2 — VERIFY ONLY: no installs, no state changes.
# Tells you (a) whether this fresh runtime's baked-in torch is already >= 2.6
# (in which case skip Cell 4 entirely), and (b) the real, live list of torch
# versions pip can currently resolve for this exact Python build/platform —
# so the next step pins something pip has actually confirmed exists here,
# rather than a version+CUDA-tag combination guessed in advance.
import subprocess
import sys

print("Python:", sys.version)
print()
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,compute_cap,memory.total", "--format=csv"],
    capture_output=True, text=True,
).stdout)

try:
    import torch
    print("Preinstalled torch:", torch.__version__)
    print("torch built for CUDA:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
except ImportError:
    print("torch not preinstalled in this image")

print()
print("--- torch versions PyPI serves right now for this runtime ---")
!pip index versions torch

## Read Cell 2's output before continuing

- **If "Preinstalled torch" is already >= 2.6:** skip Cell 4 and the restart entirely — go straight to Cell 5.
- **If it's < 2.6 (or torch isn't preinstalled):** run Cell 4, then restart the runtime (menu action below), then Cell 5.

Cell 4 deliberately does **not** pass `--index-url` or a hardcoded CUDA tag (e.g. `cu121`). That specific combination — pinning an old CUDA-tagged index while requesting `torch>=2.6` — is the most likely cause of the earlier `No matching distribution found` error: PyTorch's `cu121` index does not carry `torch>=2.6` builds (current releases are tagged `cu126`/`cu128`/`cu130`). Letting `pip install` resolve from the default index avoids hardcoding a tag that can go stale between torch releases.

In [ ]:
# Cell 4 — ONLY run this if Cell 2 showed torch < 2.6.
# No --index-url pin: let pip resolve the correct CUDA-enabled build for this
# runtime's Python/platform from the default index.
!pip install --upgrade "torch>=2.6" --quiet

## Restart runtime now (only if you ran Cell 4)

**Runtime -> Restart session**, then continue from Cell 5.

In [ ]:
# Cell 5 — Post-restart environment restore + version assertion
import os
from pathlib import Path

WORK = Path("/content/rcs_project")
os.chdir(WORK)
os.environ["PYTHONPATH"] = "src:."
os.environ["HF_HOME"] = "/content/hf_cache"
GOLD_CSV = "data/gold/gold_train_val_only.csv"

import torch
from packaging.version import Version

base_version = torch.__version__.split("+")[0]
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert Version(base_version) >= Version("2.6"), f"torch is still {torch.__version__} — Cell 4 did not take effect, or the runtime was not restarted"

In [ ]:
# Cell 6 — Strict preflight (also re-verifies val_rows==108, human_test absent
# from the CSV, taxonomy alignment; includes a MARBERTv2 forward-pass smoke
# test — inference only, no Trainer.train() call, no checkpoint written).
!python colab/preflight.py --gold-csv {GOLD_CSV}

In [ ]:
# Cell 7 — Sentiment-only inference run.
# --skip-classical --skip-transformers: only evaluate_pretrained_sentiment()
# executes. No TF-IDF baselines, no MARBERTv2/AraBERT fine-tuning.
OUTPUT = "/content/selection_phase_e_sentiment"
!python scripts/run_model_selection.py \
  --gold-csv {GOLD_CSV} \
  --output-dir {OUTPUT} \
  --skip-classical \
  --skip-transformers

In [ ]:
# Cell 8 — Package + download artifacts.
# model_selection_summary.json already embeds the resolved torch/transformers/
# cuda versions for this run via _environment_info() — no separate env-log
# code needed.
!python colab/package_artifacts.py --selection-dir /content/selection_phase_e_sentiment --output /content/rcs_phase_e_sentiment_artifacts.zip
from google.colab import files
files.download("/content/rcs_phase_e_sentiment_artifacts.zip")

## After downloading

- Expect inside the zip: `sentiment_pretrained_val_report.json` (macro-F1, per-class P/R/F1/support, confusion matrix), `sentiment_pretrained_val_probs.json` (per-row probabilities for all 108 val rows), and `model_selection_summary.json`/`.txt` (includes this run's actual `torch`/`transformers`/`cuda` versions).
- Merge these into your local `data/models/selection/` alongside the existing `tfidf_*_sentiment_val_report.json` and the Intent/Topic MARBERTv2 artifacts — keep the filenames as-is so they don't collide with the classical baseline reports already there.
- Do **not** copy anything back into the MARBERT runtime's environment or re-run `colab/setup_gpu_env.py --install` there — that runtime stays pinned at torch 2.5.1+cu121, verified and untouched.
- `human_test` was not opened at any point in this notebook (the bundle doesn't even contain `gold_dataset_round1.csv`, only the test-excluded `gold_train_val_only.csv`).